In [ ]:
# run this notebook after 22_remove_giab_from_ibd2_python
# use this notebook to generate 96-type phenotype files at the individual and family levels. 
# after this, run 24_missense_lof_python

In [ ]:
library(data.table)
library(ggplot2)
library(dplyr)

In [ ]:
trios_all = fread("./relatedness_dec20/trios_final.txt")
sibs_all = fread("./relatedness_dec20/sibs_final.txt")
sibs_all$id = paste(sibs_all$idv1, sibs_all$idv2, sep = "_")
sibs_all = sibs_all %>% select(fid, id)
trios_all$id = paste(trios_all$par1, trios_all$par2, trios_all$offspring, sep = "_")
trios_all = trios_all %>% select(fid, id)

In [ ]:
sibs_clean = fread("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt", header=TRUE)
sibs_clean$id = paste(sibs_clean$idv1, sibs_clean$idv2, sep = "_")
sibs_clean$mutant_id = sibs_clean$idv
sibs_clean = merge(sibs_clean, sibs_all, by = c("id"))
sibs_clean = sibs_clean[,c("chr", "pos", "ref", "alt", "id", "fid", "mutant_id", "Type")]
sibs_clean$source = "sib"
length(unique(sibs_clean$id))

trios = fread("./trios_snps_after_standard_QC_GIAB_COSMIC.txt")
trios$id = paste(trios$par1, trios$par2, trios$off, sep = "_")
trios$mutant_id = trios$off
trios_clean = merge(trios, trios_all, by = c("id"))
trios_clean = trios_clean[,c("chr", "pos", "ref", "alt", "id", "fid", "mutant_id", "Type")]
trios_clean$source = "trio"
length(unique(trios_clean$id))

In [ ]:
all_clean = rbind(sibs_clean, trios_clean)

all_mutants = unique(all_clean$mutant_id)

cosmic_types = unique(all_clean$Type)

all_idvs = unique(all_clean$id)

sibs_all$source = "sib"
trios_all$source = "trio"

family_to_idv = rbind(sibs_all, trios_all)

calculate_cosmic_vector = function(df){
    output = rep(NA, length(cosmic_types))
    for (j in 1:length(cosmic_types)){
        sig = cosmic_types[j]
        s = df %>% filter(Type == sig)
        output[j] = nrow(s)
    }
    return(output)
}

In [ ]:
cosmic_sib_trio = data.frame(id = all_idvs)
cosmic_sib_trio[,cosmic_types] = NA

In [ ]:
for (i in 1:length(all_idvs)){
    if (i %% 1000 == 0){
        message(i)
    }
    # identify subset
    sub = all_clean %>% filter(id == all_idvs[i])
    # calculate cosmic vector 
    vec = calculate_cosmic_vector(sub)
    cosmic_sib_trio[i,2:ncol(cosmic_sib_trio)] = vec
}
cosmic_sib_trio_fid = merge(cosmic_sib_trio, family_to_idv, by = "id")
cosmic_sib_trio_fid = unique(cosmic_sib_trio_fid)
nrow(cosmic_sib_trio_fid)


In [ ]:
# now, generate family files 
trios = fread("./relatedness_dec20/trios_final.txt", header=TRUE)
sibs =fread("./relatedness_dec20/sibs_final.txt", header=TRUE)
trios$id = paste(trios$par1, trios$par2, trios$offspring, sep = "_")
sibs$id = paste(sibs$idv1, sibs$idv2, sep = "_")
cosmic_96 = cosmic_sib_trio_fid
cosmic_types = names(cosmic_96)[2:97]
sibs = sibs %>% filter(id %in% cosmic_96$id)

In [ ]:
fwrite(cosmic_sib_trio_fid, "./sibs_trios_clean_COSMIC_96.txt", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
all_families = unique(c(trios$fid, sibs$fid))
# generate a file with families, trios, and siblings 
families = data.frame(fid = all_families)
families$ids <- vector("list", nrow(families))
families$source = NA
for (i in 1:nrow(families)){
    # subset trio to family 
    trio_fam = trios %>% filter(fid == families$fid[i])
    sib_fam = sibs %>% filter(fid == families$fid[i])
    # if trio fam exists, populate by trio only 
    if (nrow(trio_fam) != 0){
        families$ids[[i]] = trio_fam$id
        families$source[i] = "trio"
    } else {
        families$ids[[i]] = sib_fam$id
        families$source[i] = "sib"
    }
}

In [ ]:
# read in sibling denominators 
nrow(cosmic_96)
ibd2_sib = fread("./aou_ibd2_giab_removed.tsv")
names(ibd2_sib) = c("id", "denominator")
ibd2_sib$denominator= 4 * ibd2_sib$denominator
cosmic_96 = merge(cosmic_96, ibd2_sib, by = "id", all.x = TRUE)
cosmic_96 = unique(cosmic_96)
nrow(cosmic_96)
# add trio denominators 
cosmic_96$denominator[which(is.na(cosmic_96$denominator))] = 2 * 2294489308
# generate mutation rates 
cosmic_96$mutation_rate = NA
cosmic_96$tot_mutations = NA
for (i in 1:nrow(cosmic_96)){
    tot_mutations = sum(cosmic_96[i,..cosmic_types])
    cosmic_96$mutation_rate[i] = tot_mutations/cosmic_96$denominator[i]
    cosmic_96$tot_mutations[i] = tot_mutations
}

In [ ]:
c_a_names = cosmic_types[which(substr(cosmic_types, 3, 5) == "C>A")]
c_t_names = cosmic_types[which(substr(cosmic_types, 3, 5) == "C>T" & substr(cosmic_types, 3, 7) != "C>T]G")]
c_g_names = cosmic_types[which(substr(cosmic_types, 3, 5) == "C>G")]
t_a_names = cosmic_types[which(substr(cosmic_types, 3, 5) == "T>A")]
t_c_names = cosmic_types[which(substr(cosmic_types, 3, 5) == "T>C")]
t_g_names = cosmic_types[which(substr(cosmic_types, 3, 5) == "T>G")]
cpg_names = cosmic_types[which(substr(cosmic_types, 3, 7) == "C>T]G")]

In [ ]:
# generate c_a mutation rates 
genome = fread("./hg38.chrom.sizes")
genome = genome %>% filter(V1 %in% paste("chr", 1:22, sep = ""))
genome = sum(genome$V2)
c_a = 907147366/2294489308 # the number of C (or G) in the reference genome after removing GIAB regions
c_g = c_a
cpg = 39815717/2294489308 # the number of CG (or GC) in the reference genome after removing GIAB regions
c_t = c_a - cpg
t_c = 1 - c_a
t_g = t_c
t_a = t_c

cosmic_96$c_a_denominator = cosmic_96$denominator * c_a
cosmic_96$c_t_denominator = cosmic_96$denominator * c_t 
cosmic_96$c_g_denominator = cosmic_96$denominator * c_g 

cosmic_96$t_a_denominator = cosmic_96$denominator * t_a
cosmic_96$t_c_denominator = cosmic_96$denominator * t_c 
cosmic_96$t_g_denominator = cosmic_96$denominator * t_g 

cosmic_96$cpg_denominator = cosmic_96$denominator * cpg 

cosmic_96$c_a_rate = NA
cosmic_96$cpg_rate = NA
cosmic_96$c_t_rate = NA
cosmic_96$c_g_rate = NA
cosmic_96$t_c_rate = NA
cosmic_96$t_a_rate = NA
cosmic_96$t_g_rate = NA

for (i in 1:nrow(cosmic_96)){
    cosmic_96$c_a_rate[i] = sum(unname(unlist((cosmic_96[i, ..c_a_names]))))/cosmic_96$c_a_denominator[i]
    cosmic_96$c_t_rate[i] = sum(unname(unlist((cosmic_96[i, ..c_t_names]))))/cosmic_96$c_t_denominator[i]
    cosmic_96$c_g_rate[i] = sum(unname(unlist((cosmic_96[i, ..c_g_names]))))/cosmic_96$c_g_denominator[i]
    cosmic_96$cpg_rate[i] =  sum(unname(unlist((cosmic_96[i, ..cpg_names]))))/cosmic_96$cpg_denominator[i]
    cosmic_96$t_g_rate[i] = sum(unname(unlist((cosmic_96[i, ..t_g_names]))))/cosmic_96$t_g_denominator[i]
    cosmic_96$t_a_rate[i] = sum(unname(unlist((cosmic_96[i, ..t_a_names]))))/cosmic_96$t_a_denominator[i]
    cosmic_96$t_c_rate[i] = sum(unname(unlist((cosmic_96[i, ..t_c_names]))))/cosmic_96$t_c_denominator[i]
}

                                

In [ ]:
calculate_cosmic_vector = function(df){
    output = rep(NA, length(cosmic_types))
    for (j in 1:length(cosmic_types)){
        sig = cosmic_types[j]
        s = df %>% filter(Type == sig)
        output[j] = nrow(s)
    }
    return(output)
}

In [ ]:
sibling_mutations = fread("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt", header=TRUE)
sibling_mutations$pair = paste(sibling_mutations$idv1, sibling_mutations$idv2, sep = "_")

In [ ]:
# generate combined 96 type tables 
# generate average mutation rate 
families$mutation_rates = vector("list", nrow(families))
families$average_mutation_rate = NA
families[cosmic_types] = NA
families$tot_mutations = vector("list", nrow(families))
families$denominators = vector("list", nrow(families))
families$c_a_denominators = vector("list", nrow(families))
families$c_a_rates = vector("list", nrow(families))
families$cpg_denominators = vector("list", nrow(families))
families$cpg_rates = vector("list", nrow(families))
families$tot_mutations_cosmic = NA
for (i in 1:nrow(families)){
    ids = families$ids[[i]]
    idx = which(cosmic_96$id %in% ids)
    # if trio, simply sum the 96-type mutations 
    if (families$source[i] == "trio"){
       families[i,cosmic_types] = unname(colSums(cosmic_96[idx, ..cosmic_types])) 
    } else { # if sibling, read in mutations, subset to unique ones, calculate 96-type vectors from the union
        # of all mutations
        # all mutations belonging to this family 
        sibling_sub = sibling_mutations %>% filter(pair %in% ids)
        # all unique mutations 
        sibling_sub = unique(sibling_sub %>% select(chr, pos, ref, alt, idv, Type))
        # calculate 96-type and assign
        families[i,cosmic_types] = calculate_cosmic_vector(sibling_sub)
    }
    families$tot_mutations_cosmic[i] = sum(unname((cosmic_96[idx, ..cosmic_types])))
    families[[i,"tot_mutations"]] = unname(unlist(cosmic_96[idx, "tot_mutations"]))
    families[[i,"mutation_rates"]] = unname(unlist(cosmic_96[idx, "mutation_rate"]))
    families[[i,"c_a_rates"]] = unname(unlist(cosmic_96[idx, "c_a_rate"]))
    families[[i,"cpg_rates"]] = unname(unlist(cosmic_96[idx, "cpg_rate"]))
    families[[i,"c_a_denominators"]] = unname(unlist(cosmic_96[idx, "c_a_denominator"]))
    families[[i,"cpg_denominators"]] = unname(unlist(cosmic_96[idx, "cpg_denominator"]))
    families[i,"average_mutation_rate"] = mean(unname(unlist(cosmic_96[idx, "mutation_rate"])))
    families[[i,"denominators"]] = unname(unlist(cosmic_96[idx, "denominator"]))
}



In [ ]:
# generate combined 96 type tables 
# generate average mutation rate 
families$mutation_rates = vector("list", nrow(families))
families$average_mutation_rate = NA
families[cosmic_types] = NA
families$tot_mutations = vector("list", nrow(families))
families$denominators = vector("list", nrow(families))

families$c_a_denominators = vector("list", nrow(families))
families$c_a_rates = vector("list", nrow(families))

families$cpg_denominators = vector("list", nrow(families))
families$cpg_rates = vector("list", nrow(families))

families$c_g_denominators = vector("list", nrow(families))
families$c_g_rates = vector("list", nrow(families))

families$c_t_denominators = vector("list", nrow(families))
families$c_t_rates = vector("list", nrow(families))

families$t_c_denominators = vector("list", nrow(families))
families$t_c_rates = vector("list", nrow(families))

families$t_a_denominators = vector("list", nrow(families))
families$t_a_rates = vector("list", nrow(families))

families$t_g_denominators = vector("list", nrow(families))
families$t_g_rates = vector("list", nrow(families))

families$tot_mutations_cosmic = NA
for (i in 1:nrow(families)){
    ids = families$ids[[i]]
    idx = which(cosmic_96$id %in% ids)
    # if trio, simply sum the 96-type mutations 
    if (families$source[i] == "trio"){
       families[i,cosmic_types] = unname(colSums(cosmic_96[idx, ..cosmic_types])) 
    } else { # if sibling, read in mutations, subset to unique ones, calculate 96-type vectors from the union
        # of all mutations
        # all mutations belonging to this family 
        sibling_sub = sibling_mutations %>% filter(pair %in% ids)
        # all unique mutations 
        sibling_sub = unique(sibling_sub %>% select(chr, pos, ref, alt, idv, Type))
        # calculate 96-type and assign
        families[i,cosmic_types] = calculate_cosmic_vector(sibling_sub)
    }
    families$tot_mutations_cosmic[i] = sum(unname((cosmic_96[idx, ..cosmic_types])))
    families[[i,"tot_mutations"]] = unname(unlist(cosmic_96[idx, "tot_mutations"]))
    families[[i,"mutation_rates"]] = unname(unlist(cosmic_96[idx, "mutation_rate"]))
    
    families[[i,"c_a_rates"]] = unname(unlist(cosmic_96[idx, "c_a_rate"]))
    families[[i,"c_a_denominators"]] = unname(unlist(cosmic_96[idx, "c_a_denominator"]))
    
    families[[i,"c_t_rates"]] = unname(unlist(cosmic_96[idx, "c_t_rate"]))
    families[[i,"c_t_denominators"]] = unname(unlist(cosmic_96[idx, "c_t_denominator"]))
        
    families[[i,"c_g_rates"]] = unname(unlist(cosmic_96[idx, "c_g_rate"]))
    families[[i,"c_g_denominators"]] = unname(unlist(cosmic_96[idx, "c_g_denominator"]))
    
    families[[i,"t_a_rates"]] = unname(unlist(cosmic_96[idx, "t_a_rate"]))
    families[[i,"t_a_denominators"]] = unname(unlist(cosmic_96[idx, "t_a_denominator"]))
    
    families[[i,"t_c_rates"]] = unname(unlist(cosmic_96[idx, "t_c_rate"]))
    families[[i,"t_c_denominators"]] = unname(unlist(cosmic_96[idx, "t_c_denominator"]))
        
    families[[i,"t_g_rates"]] = unname(unlist(cosmic_96[idx, "t_g_rate"]))
    families[[i,"t_g_denominators"]] = unname(unlist(cosmic_96[idx, "t_g_denominator"]))
    
    families[[i,"cpg_rates"]] = unname(unlist(cosmic_96[idx, "cpg_rate"]))
    families[[i,"cpg_denominators"]] = unname(unlist(cosmic_96[idx, "cpg_denominator"]))
    
    families[i,"average_mutation_rate"] = mean(unname(unlist(cosmic_96[idx, "mutation_rate"])))
    families[[i,"denominators"]] = unname(unlist(cosmic_96[idx, "denominator"]))
}


In [ ]:
# now read in pcs and add 
families[paste("pc", 1:6, sep = "")] = NA
pcs_to_add = paste("pc", 1:6, sep = "")
# we will add the pc of a random offspring 
for (i in 1:nrow(families)){
    if (families$source[i] == "trio"){
        # choose a random offspring from the family 
        random_offspring = sample(as.character(trios$offspring[which(trios$fid == families$fid[i])]),1)
    }
    else {
       # choose a random offspring from the family
        all_offspring = c(sibs$idv1[which(sibs$fid == families$fid[i])],
                          sibs$idv2[which(sibs$fid == families$fid[i])])
        random_offspring = sample(as.character(all_offspring),1)
    }
    # populate pcs 
    families[i,pcs_to_add] = pcs[which(pcs$s == random_offspring), ..pcs_to_add] 
}


In [ ]:
# now, write the 96-type in family format 
fwrite(families, "./families_COSMIC_96_mutation_rates.txt", sep = "\t", 
      row.names = FALSE, quote = FALSE)